# SOMA — Ablation: Necessity Signals

**Wakasa Labs · Nairobi, Kenya · March 2026**

Removes each necessity signal (N1, N2, N3) one at a time to validate
that each contributes to the overall BT performance.

This produces **Table 2** of Paper 1.

Expected results:
- Full SOMA: BT ≈ -0.038, K ≈ 7
- No N1: BT ≈ -0.065, K ≈ 9 (over-spawns on lr-decay plateaus)
- No N2: BT ≈ -0.071, K ≈ 10 (doesn't detect subspace gaps)
- No N3: BT ≈ -0.058, K ≈ 8 (triggers on noise)
- No RL: BT ≈ -0.047, K ≈ 8 (close but less efficient)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
import sys
import os
import argparse
import pandas as pd

# Add parent directory to path to import soma
sys.path.insert(0, os.path.abspath('..'))

from soma.experiments.run_permuted_mnist import run_experiment

base_args = {
    "n_tasks": 10,
    "n_train": 1000,
    "n_test": 200,
    "device": device,
    "seed": 42,
    "no_rl": False,
    "disable_n1": False,
    "disable_n2": False,
    "disable_n3": False,
}

variants = [
    ("Full SOMA", {}),
    ("No N1", {"disable_n1": True}),
    ("No N2", {"disable_n2": True}),
    ("No N3", {"disable_n3": True}),
    ("No RL", {"no_rl": True}),
]

results = []

print("Running ablation experiments...")
for name, kwargs in variants:
    print(f"\n--- {name} ---")
    current_args = base_args.copy()
    current_args.update(kwargs)
    args = argparse.Namespace(**current_args)

    result = run_experiment(args)
    
    results.append({
        "Variant": name,
        "BT": result['backward_transfer'],
        "K": result['final_k']
    })

print("\n--- Ablation Results (Table 2) ---")
df = pd.DataFrame(results)
print(df.to_string(index=False))